## <font color=yellow>ABC News </font>
- 토픽 모델링
    - 뉴스 헤드라인을 이용하면 같은 토픽끼리 묶을 수 있지 않을까에서 시작
    - clutering (각 문서가 유사하다고 생각하는 것 끼리 묶음)
    - 정보의 다양성 측정
    - outlier detection

==>VectorDB에 저장하고자 하는 컨텐츠에 대한 검수 및 전처리

In [ ]:
import pandas as pd
import os
import json
import openai
from openai import OpenAI
import numpy as np
from tqdm.notebook import tqdm, trange
from sklearn.cluster import KMeans
from utils import create_embeddings

# initialize openai
os.environ['OPENAI_API_KEY']= ""
openai.api_key = os.environ["OPENAI_API_KEY"]

In [3]:
df = pd.read_csv("abcnews_2020.csv")
df.head()

,publish_date,headline_text
0,20200101,a new type of resolution for the new year
1,20200101,adelaide records driest year in more than a de...
2,20200101,adelaide riverbank catches alight after new ye...
3,20200101,adelaides 9pm fireworks spark blaze on riverbank
4,20200101,archaic legislation governing nt women propert...


## <font color=yellow>1. Clustering </font>

In [4]:
# 비용 발생 주의

batch_size = 2000
headline_emb = list()

headline = df['headline_text'].tolist()

for i in trange(0, len(headline), batch_size):
    i_end = min(len(headline), i+batch_size)
    data_batch = headline[i:i_end]

    tmp_emb = create_embeddings(data_batch)
    headline_emb.extend(tmp_emb)

  0%|          | 0/2 [00:00<?, ?it/s]

trange는 Python에서 tqdm 라이브러리의 기능 중 하나로, for 루프에서 진행 상황(progress bar) 을 시각적으로 보여주는 데 사용

In [5]:
df['headline_emb'] = headline_emb

In [6]:
df.head()

,publish_date,headline_text,headline_emb
0,20200101,a new type of resolution for the new year,"[-0.02990344539284706, 0.027576573193073273, 0..."
1,20200101,adelaide records driest year in more than a de...,"[0.023331310600042343, 0.024199753999710083, 0..."
2,20200101,adelaide riverbank catches alight after new ye...,"[0.008516565896570683, -0.006767496466636658, ..."
3,20200101,adelaides 9pm fireworks spark blaze on riverbank,"[0.031862929463386536, 1.1516153790580574e-05,..."
4,20200101,archaic legislation governing nt women propert...,"[0.05418633669614792, 0.061872921884059906, 0...."


In [7]:
df.to_csv("abcnews_2020_emb.csv", index=False)
df = pd.read_csv("abcnews_2020_emb.csv")
df.head()

,publish_date,headline_text,headline_emb
0,20200101,a new type of resolution for the new year,"[-0.02990344539284706, 0.027576573193073273, 0..."
1,20200101,adelaide records driest year in more than a de...,"[0.023331310600042343, 0.024199753999710083, 0..."
2,20200101,adelaide riverbank catches alight after new ye...,"[0.008516565896570683, -0.006767496466636658, ..."
3,20200101,adelaides 9pm fireworks spark blaze on riverbank,"[0.031862929463386536, 1.1516153790580574e-05,..."
4,20200101,archaic legislation governing nt women propert...,"[0.05418633669614792, 0.061872921884059906, 0...."


In [8]:
type(df.loc[0, 'headline_emb'])

str

저장하고 다시 불러들어올때 string값으로 되어 있음
- 지난 시간에 이야기 했음

In [9]:
df['headline_emb'] = df['headline_emb'].apply(json.loads)
type(df.loc[0, 'headline_emb'])

list

In [10]:
clusters = KMeans(n_clusters=15, random_state=0).fit_predict(df['headline_emb'].tolist())
df['cluster'] = clusters

In [11]:
df.head()

,publish_date,headline_text,headline_emb,cluster
0,20200101,a new type of resolution for the new year,"[-0.02990344539284706, 0.027576573193073273, 0...",2
1,20200101,adelaide records driest year in more than a de...,"[0.023331310600042343, 0.024199753999710083, 0...",11
2,20200101,adelaide riverbank catches alight after new ye...,"[0.008516565896570683, -0.006767496466636658, ...",9
3,20200101,adelaides 9pm fireworks spark blaze on riverbank,"[0.031862929463386536, 1.1516153790580574e-05,...",9
4,20200101,archaic legislation governing nt women propert...,"[0.05418633669614792, 0.061872921884059906, 0....",8


In [12]:
df.loc[df['cluster']==1]

,publish_date,headline_text,headline_emb,cluster
162,20200103,nick kyrgios kicks off australias atp cup chal...,"[-0.05597231909632683, 0.01846293918788433, 0....",1
201,20200104,bushfire help sparked by ashleigh barty pink a...,"[-0.0062495265156030655, -0.032479289919137955...",1
249,20200104,wrong anthem played for moldova at atp cup,"[-0.03378334268927574, 0.012585321441292763, 0...",1
297,20200105,sasha zhoya turns back on australia athletics ...,"[0.026829250156879425, 0.0323784202337265, 0.0...",1
302,20200105,stars marcus stoinis fined personal abuse rene...,"[-0.02918173372745514, 0.06353995203971863, 0....",1
...,...,...,...,...
2314,20200130,rafael nadal agitated by chair umpire after gi...,"[-0.051331765949726105, 0.03031117469072342, 0...",1
2315,20200130,rafael nadal loses to dominic thiem australian...,"[-0.018239960074424744, 0.014115291647613049, ...",1
2354,20200131,australian open dominic thiem beats alexander ...,"[-0.010364953428506851, 0.03771139308810234, 0...",1
2355,20200131,australian open has delivered more that we cou...,"[-0.018940100446343422, 0.05168469622731209, 0...",1


## <font color=yellow>2. 정보의 다양성 (Diversity) 측정 </font>
- clutering이 잘 되었을까?
- 한 cluster 안에 있더라도 정보가 다양할까?

==> 이를 측정할 수 있는 방법은 역시 임베딩

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_diversity(df, column_name):
    """
    Calculates the diversity of a set of embeddings based on cosine distance.
    
    :param embeddings: NumPy array of embeddings
    :return: The average cosine distance between embeddings, higher means more diverse
    """
    # 각각의 임베딩끼리 모두 pairwise cosine similarity를 계산
    embeddings = np.vstack(df[column_name])
    cosine_sim = cosine_similarity(embeddings)
    
    # self-comparisons (diagonal elements)를 제외하고 cosine similarity 계산
    np.fill_diagonal(cosine_sim, np.nan) # 본인과의 similarity는 제외
    avg_distance = np.nanmean(cosine_sim)
    
    return cosine_sim, avg_distance

각 Cluster안에 있는 임베딩 값들을, 서로 cosine similarity를 다 계산 해줌 
- 모두 계산된 cosine similarity 평균을 구함 
- 1이면 정보가 비슷한 것이고, 0에 가까울 수록 정보가 다양한 것임

In [14]:
dist, avg = calculate_diversity(df, 'headline_emb')

In [16]:
dist

array([[       nan, 0.17473013, 0.30675202, ..., 0.05150858, 0.21780998,
        0.09137895],
       [0.17473013,        nan, 0.51811767, ..., 0.03843003, 0.19514557,
        0.07771653],
       [0.30675202, 0.51811767,        nan, ..., 0.07019599, 0.12073827,
        0.09891301],
       ...,
       [0.05150858, 0.03843003, 0.07019599, ...,        nan, 0.05116549,
        0.31273248],
       [0.21780998, 0.19514557, 0.12073827, ..., 0.05116549,        nan,
        0.07625165],
       [0.09137895, 0.07771653, 0.09891301, ..., 0.31273248, 0.07625165,
               nan]])

각각 클러스터 안에 있는 정보들은 얼마나 다양할지 봐보자
- 위는 클러스터 간의 cosine similarity를 구한 것임

pjt에 적용한다면 문서가 여러개 있으면, 문서를 여러 단위로 chunking하고 코레스파운딩? 하는 청킹들 사이에 유사도를 구하고, 그 다음에 그렇게 구해진 유사도를 전체적으로 평균을 냄 (혹은 가중평균)
- 이런식으로 문서의 다양성, 유사성을 판단할 수 있음

In [15]:
avg

0.19836089662662126

1. 문서 여러 개를 준비
- (예: 뉴스 기사, 블로그, 보고서 등)

2. 각 문서를 여러 개의 청크(chunk)로 나눔
- (예: 문단 단위, 문장 N개 단위 등)

3. 각 청크를 임베딩(embedding)
- (예: BERT, OpenAI Embedding 등)

4. 문서 간 비교 방식:
- 한 문서 A의 모든 청크 vs 문서 B의 모든 청크 간 pairwise 유사도 계산
- 그 결과로 두 문서 간 유사도 스코어를 도출
    - (예: 평균, 최대값, 가중 평균 등 다양한 방식)

5. 전체 문서 집합에 대해 위 과정을 반복해서
- 문서 간 평균 유사도 → 전체 유사성 수준
- 또는 1 - 평균 유사도 → 전체 다양성 지표로 사용 가능

In [17]:
diversity_score = {k:calculate_diversity(df.loc[df['cluster']==k], 'headline_emb')[1] for k in range(0, 15)}
diversity_score

{0: 0.269347526782569,
 1: 0.4270107210609911,
 2: 0.15159416618069815,
 3: 0.39409017374500466,
 4: 0.2908405012108005,
 5: 0.5788221967106328,
 6: 0.4741100116954729,
 7: 0.2841194143321932,
 8: 0.29025110573268664,
 9: 0.46801957942849653,
 10: 0.19148008600425953,
 11: 0.40571701261162907,
 12: 0.38507854841417133,
 13: 0.5049384379180334,
 14: 0.28638597428551776}

In [19]:
df.loc[df['cluster']==3]

,publish_date,headline_text,headline_emb,cluster
212,20200104,china to identify cause of mystery pneumonia p...,"[-0.006433676462620497, -0.0543876476585865, 0...",3
259,20200105,china replace top official in hong kong as pro...,"[-0.013344322331249714, -0.021175269037485123,...",3
360,20200106,mysterious illness in china is not sars,"[-0.0005243383930064738, -0.033341310918331146...",3
516,20200108,china australia relations became complex in 2019,"[-0.04726170375943184, -0.005923602730035782, ...",3
632,20200109,measles outbreak kills 6000 people in congo,"[0.020234717056155205, -0.06365872174501419, 0...",3
...,...,...,...,...
2437,20200131,wall street volatile coronavirus australian do...,"[-0.07781743258237839, -0.02185390144586563, 0...",3
2442,20200131,who coronavirus global emergency,"[-0.025180980563163757, -0.004652089439332485,...",3
2443,20200131,who declares coronavirus outbreak as global he...,"[-0.04880526661872864, -0.03392207622528076, 0...",3
2444,20200131,will travel insurance cover trip cancelled ove...,"[-0.017240509390830994, -0.02051793783903122, ...",3


## <font color=yellow>3. 정보의 다양성 (Diversity) 측정 </font>
- 정보의 유사성을 측정했다면, 그 다음엔 클러스터 안에 있는 정보 중에서도 outlier를 판별해보자

In [20]:
from sklearn.ensemble import IsolationForest

In [21]:
cluster = df.loc[df['cluster']==3]

In [22]:
iso_forest = IsolationForest(contamination=0.05)  # Adjust contamination as needed
anomalies = iso_forest.fit_predict(cluster['headline_emb'].tolist())

anomalous_headlines = np.array(cluster['headline_text'].tolist())[anomalies == -1]
# print("Anomalous Headlines:", anomalous_headlines)

In [23]:
anomalies

array([ 1, -1,  1,  1, -1,  1,  1,  1, -1,  1,  1, -1,  1,  1,  1,  1,  1,
        1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1])

In [24]:
anomalous_headlines

array(['china replace top official in hong kong as protests continue',
       'measles outbreak kills 6000 people in congo',
       'hong kong airline apologises for woman take pregnancy test',
       'chinas propaganda blitz against uyghurs',
       'video shows medics checking passengers temperatures on flight',
       'government surgical mask greg hunt coronavirus',
       'will travel insurance cover trip cancelled over coronavirus'],
      dtype='<U64')

단순히 텍스트를 embedding화 하는 것에서 더 나아가, <br>
텍스트를 특징별로 묶거나 유관하지 않다고 판단되는 텍스트는 제외하는 등, 컨텐츠 자체를 preprocessing/탐색 하는데에 활용 가능
- 이런 머신러닝 모델을 사용한다면, plain한 모델을 사용하지말고, 모델 자체를 조금 파인튜닝 한다던지, 조금더 복잡하고 고도화된 모델을 활용을 해서 이런 절차를 진행하면 됨